# Laboratório — Bagging e Random Forest

Vamos investigar três hipóteses em dados tabulares sintéticos:

1. um bootstrap contém aproximadamente 63,2% de unidades distintas;
2. bagging e Random Forest reduzem a instabilidade de uma árvore;
3. a curva de desempenho satura conforme cresce o número de árvores.

O teste é separado antes de qualquer seleção e aberto uma única vez no fim. Seed global: `20260908`.

## Dependências

```text
numpy>=1.26
pandas>=2.2
matplotlib>=3.8
scikit-learn>=1.4
```

O notebook não baixa dados e não usa rede, credenciais ou serviços externos.

In [ ]:
# %pip install "numpy>=1.26" "pandas>=2.2" "matplotlib>=3.8" "scikit-learn>=1.4"
import platform
import warnings

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
from sklearn.datasets import make_classification
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import BaggingClassifier, RandomForestClassifier
from sklearn.metrics import accuracy_score, log_loss
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_validate, train_test_split
from sklearn.tree import DecisionTreeClassifier

SEED = 20260908
np.random.seed(SEED)
warnings.filterwarnings("error")
print("Python:", platform.python_version())
print("NumPy:", np.__version__)
print("pandas:", pd.__version__)
print("matplotlib:", matplotlib.__version__)
print("scikit-learn:", sklearn.__version__)

## 1. Bootstrap: linhas sorteadas versus unidades distintas

Simulamos 500 bootstraps de 1.000 unidades. Cada réplica tem 1.000 posições, mas repete índices. A teoria prevê fração única `1 - exp(-1)` e fração OOB `exp(-1)`.

In [ ]:
rng = np.random.default_rng(SEED)
n = 1000
unique_fractions = []
for _ in range(500):
    indices = rng.integers(0, n, size=n)
    unique_fractions.append(np.unique(indices).size / n)

observed_unique = float(np.mean(unique_fractions))
theoretical_unique = 1.0 - np.exp(-1)
observed_oob = 1.0 - observed_unique
assert abs(observed_unique - theoretical_unique) < 0.005
print(f"Fração única simulada: {observed_unique:.6f}")
print(f"Fração única teórica:  {theoretical_unique:.6f}")
print(f"Fração OOB simulada:   {observed_oob:.6f}")

## 2. Dados e teste lacrado

Geramos um problema equilibrado com interações, features informativas, redundantes e ruído. Cada linha é uma unidade independente. O teste recebe 25% e permanece intocado até a última seção.

In [ ]:
X, y = make_classification(
    n_samples=1800,
    n_features=20,
    n_informative=8,
    n_redundant=6,
    n_repeated=0,
    n_clusters_per_class=3,
    class_sep=1.0,
    flip_y=0.06,
    random_state=SEED,
)
X_dev, X_test, y_dev, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=SEED
)
X_subtrain, X_check, y_subtrain, y_check = train_test_split(
    X_dev, y_dev, test_size=0.22, stratify=y_dev, random_state=SEED + 1
)
assert X_dev.shape == (1350, 20)
assert X_test.shape == (450, 20)
print("Desenvolvimento:", X_dev.shape, "| teste reservado:", X_test.shape)
print("Subtreino interno:", X_subtrain.shape, "| checagem interna:", X_check.shape)
print("Proporção positiva no desenvolvimento:", f"{y_dev.mean():.4f}")

## 3. Bagging implementado explicitamente

Treinamos 40 árvores em bootstraps produzidos por um gerador controlado. A previsão do ensemble é a média das probabilidades individuais. Esta implementação didática não substitui as otimizações da biblioteca.

In [ ]:
manual_rng = np.random.default_rng(SEED + 2)
manual_trees = []
manual_probabilities = []
for b in range(40):
    idx = manual_rng.integers(0, len(X_subtrain), size=len(X_subtrain))
    tree = DecisionTreeClassifier(random_state=SEED + b)
    tree.fit(X_subtrain[idx], y_subtrain[idx])
    manual_trees.append(tree)
    manual_probabilities.append(tree.predict_proba(X_check)[:, 1])

manual_matrix = np.vstack(manual_probabilities)
manual_mean_probability = manual_matrix.mean(axis=0)
manual_prediction = (manual_mean_probability >= 0.5).astype(int)
manual_accuracy = accuracy_score(y_check, manual_prediction)
single_accuracy = accuracy_score(y_check, manual_trees[0].predict(X_check))
assert manual_matrix.shape == (40, len(X_check))
assert np.all((manual_mean_probability >= 0) & (manual_mean_probability <= 1))
print(f"Árvore 1 na checagem: {single_accuracy:.6f}")
print(f"Bagging manual:       {manual_accuracy:.6f}")

A execução isolada não prova superioridade universal. Ela apenas confirma que a agregação foi implementada e produz uma previsão probabilística válida.

## 4. Comparação justa nos mesmos folds

Dummy, árvore, bagging e Random Forest recebem os mesmos cinco folds estratificados. A métrica é acurácia porque as classes são equilibradas e o `oob_score_` padrão usa acurácia.

In [ ]:
cv = StratifiedKFold(n_splits=4, shuffle=True, random_state=SEED)
models = {
    "dummy": DummyClassifier(strategy="prior"),
    "arvore": DecisionTreeClassifier(random_state=SEED),
    "bagging": BaggingClassifier(
        estimator=DecisionTreeClassifier(random_state=SEED),
        n_estimators=100,
        bootstrap=True,
        random_state=SEED,
        n_jobs=1,
    ),
    "random_forest": RandomForestClassifier(
        n_estimators=100,
        max_features="sqrt",
        random_state=SEED,
        n_jobs=1,
    ),
}
rows = []
for name, model in models.items():
    scores = cross_validate(model, X_dev, y_dev, cv=cv, scoring="accuracy", return_train_score=True, n_jobs=1)
    rows.append({
        "modelo": name,
        "treino": scores["train_score"].mean(),
        "cv_media": scores["test_score"].mean(),
        "cv_dp": scores["test_score"].std(ddof=1),
    })
comparison = pd.DataFrame(rows).sort_values("cv_media", ascending=False)
assert comparison.loc[comparison.modelo == "random_forest", "cv_media"].iloc[0] > 0.80
print(comparison.round(5).to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
ordered = comparison.sort_values("cv_media")
ax.barh(ordered.modelo, ordered.cv_media, xerr=ordered.cv_dp, alpha=0.8, capsize=4)
ax.axvline(0.5, color="black", linestyle="--", linewidth=1, label="referência aleatória")
ax.set(xlabel="acurácia média nos folds", title="Comparação nos mesmos folds")
ax.set_xlim(0.45, 1.0)
ax.grid(axis="x", alpha=0.25)
ax.legend()
plt.show()

**Descrição do gráfico:** barras horizontais com média e desvio-padrão mostram a árvore única, bagging e Random Forest acima do baseline. A comparação usa os mesmos folds para reduzir variação alheia ao modelo.

## 5. Seleção de diversidade e regularização

Escolhemos `max_features` e `min_samples_leaf` apenas no desenvolvimento. `n_estimators=100` é mantido fixo durante a busca para que o número de árvores não se torne mais uma dimensão oportunista.

In [ ]:
grid = {
    "max_features": ["sqrt", 0.5, 1.0],
    "min_samples_leaf": [1, 3, 8],
}
search = GridSearchCV(
    RandomForestClassifier(
        n_estimators=100,
        bootstrap=True,
        oob_score=True,
        random_state=SEED,
        n_jobs=1,
    ),
    param_grid=grid,
    scoring="accuracy",
    cv=cv,
    n_jobs=1,
    return_train_score=True,
)
search.fit(X_dev, y_dev)
selected_forest = search.best_estimator_
best_cv = search.best_score_
assert best_cv > 0.80
print("Melhores parâmetros:", search.best_params_)
print(f"Melhor CV: {best_cv:.6f}")
print(f"OOB após reajuste: {selected_forest.oob_score_:.6f}")

In [ ]:
results = pd.DataFrame(search.cv_results_).sort_values("rank_test_score")
cols = ["param_max_features", "param_min_samples_leaf", "mean_train_score",
        "mean_test_score", "std_test_score", "rank_test_score"]
print(results[cols].round(5).to_string(index=False))

O OOB acima é calculado no reajuste sobre todo o desenvolvimento. Ele é comparável à CV neste caso i.i.d. simples, mas não é a mesma estimativa nem substitui o teste.

## 6. Correlação entre árvores

Comparamos as correlações das probabilidades individuais do bagging manual e de uma floresta com sorteio de features. Usamos apenas a checagem interna.

In [ ]:
forest_internal = RandomForestClassifier(
    n_estimators=40,
    max_features="sqrt",
    min_samples_leaf=1,
    random_state=SEED + 3,
    n_jobs=1,
).fit(X_subtrain, y_subtrain)
forest_matrix = np.vstack([
    estimator.predict_proba(X_check)[:, 1] for estimator in forest_internal.estimators_
])

def mean_off_diagonal_correlation(prediction_matrix):
    corr = np.corrcoef(prediction_matrix)
    upper = corr[np.triu_indices_from(corr, k=1)]
    return float(np.mean(upper))

bagging_corr = mean_off_diagonal_correlation(manual_matrix)
forest_corr = mean_off_diagonal_correlation(forest_matrix)
assert np.isfinite(bagging_corr) and np.isfinite(forest_corr)
print(f"Correlação média — bagging: {bagging_corr:.6f}")
print(f"Correlação média — floresta: {forest_corr:.6f}")

O sorteio de features busca reduzir correlação sem destruir a força individual. O valor depende do dataset, da métrica de correlação e da configuração; não é uma garantia de que toda floresta terá correlação menor em qualquer execução.

## 7. Saturação do número de árvores

Usamos a configuração selecionada e variamos apenas `n_estimators` na divisão interna. Para evitar avisos e estimativas OOB frágeis, calculamos OOB somente a partir de 25 árvores.

In [ ]:
sizes = [1, 5, 10, 25, 50, 100, 200]
saturation_rows = []
for size in sizes:
    model = RandomForestClassifier(
        n_estimators=size,
        max_features=search.best_params_["max_features"],
        min_samples_leaf=search.best_params_["min_samples_leaf"],
        bootstrap=True,
        oob_score=size >= 25,
        random_state=SEED,
        n_jobs=1,
    ).fit(X_subtrain, y_subtrain)
    saturation_rows.append({
        "n_estimators": size,
        "check_accuracy": accuracy_score(y_check, model.predict(X_check)),
        "oob_accuracy": model.oob_score_ if size >= 25 else np.nan,
    })
saturation = pd.DataFrame(saturation_rows)
assert saturation.check_accuracy.iloc[-1] > 0.80
print(saturation.round(5).to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(saturation.n_estimators, saturation.check_accuracy, "o-", label="checagem interna")
ax.plot(saturation.n_estimators, saturation.oob_accuracy, "s--", label="OOB")
ax.set(xlabel="número de árvores", ylabel="acurácia", title="Curva de saturação")
ax.set_xscale("log")
ax.grid(alpha=0.25)
ax.legend()
plt.show()

**Descrição do gráfico:** a acurácia oscila muito com poucas árvores e se estabiliza progressivamente. Depois da região de saturação, aumentar o ensemble altera pouco a métrica, embora o custo continue crescendo.

## 8. Estabilidade entre seeds

Na mesma divisão interna, repetimos uma árvore e uma floresta com dez seeds. Isso mede sensibilidade ao sorteio/ajuste naquele conjunto; não mede drift nem incerteza fora do domínio.

In [ ]:
tree_scores, forest_scores = [], []
for offset in range(10):
    tree = DecisionTreeClassifier(random_state=SEED + offset).fit(X_subtrain, y_subtrain)
    forest = RandomForestClassifier(
        n_estimators=100,
        max_features=search.best_params_["max_features"],
        min_samples_leaf=search.best_params_["min_samples_leaf"],
        random_state=SEED + offset,
        n_jobs=1,
    ).fit(X_subtrain, y_subtrain)
    tree_scores.append(accuracy_score(y_check, tree.predict(X_check)))
    forest_scores.append(accuracy_score(y_check, forest.predict(X_check)))

tree_sd = float(np.std(tree_scores, ddof=1))
forest_sd = float(np.std(forest_scores, ddof=1))
assert forest_sd < tree_sd
print(f"Árvore:   média={np.mean(tree_scores):.6f}, dp={tree_sd:.6f}")
print(f"Floresta: média={np.mean(forest_scores):.6f}, dp={forest_sd:.6f}")
print(f"Razão dp árvore/floresta: {tree_sd / forest_sd:.3f}")

## 9. Importância por impureza como diagnóstico

Exibimos o ranking do ajuste final, mas não o usamos para selecionar features nem fazemos afirmação causal. As features sintéticas não têm nomes semânticos.

In [ ]:
importance = pd.Series(
    selected_forest.feature_importances_,
    index=[f"feature_{i:02d}" for i in range(X_dev.shape[1])],
).sort_values(ascending=False)
assert np.isclose(importance.sum(), 1.0)
print(importance.head(10).to_frame("importancia_MDI").round(5).to_string())

Features correlacionadas podem dividir ou substituir importância; features com muitos candidatos de corte podem ser favorecidas. A Aula 20 tratará interpretação com protocolos específicos.

## 10. Avaliação final única

Congelamos três comparadores: árvore irrestrita, bagging com 100 árvores e a floresta selecionada. Todos são ajustados no desenvolvimento completo; então o teste é aberto uma única vez.

In [ ]:
final_models = {
    "arvore": DecisionTreeClassifier(random_state=SEED).fit(X_dev, y_dev),
    "bagging": BaggingClassifier(
        estimator=DecisionTreeClassifier(random_state=SEED),
        n_estimators=100,
        random_state=SEED,
        n_jobs=1,
    ).fit(X_dev, y_dev),
    "random_forest": selected_forest,
}
final_rows = []
for name, model in final_models.items():
    probability = model.predict_proba(X_test)
    prediction = np.argmax(probability, axis=1)
    final_rows.append({
        "modelo": name,
        "test_accuracy": accuracy_score(y_test, prediction),
        "test_log_loss": log_loss(y_test, probability, labels=[0, 1]),
    })
final_results = pd.DataFrame(final_rows).sort_values("test_accuracy", ascending=False)
rf_test_accuracy = final_results.loc[final_results.modelo == "random_forest", "test_accuracy"].iloc[0]
tree_test_accuracy = final_results.loc[final_results.modelo == "arvore", "test_accuracy"].iloc[0]
assert rf_test_accuracy > 0.80
assert rf_test_accuracy >= tree_test_accuracy
print(final_results.round(6).to_string(index=False))

## Conclusão auditável

- A fração de unidades distintas do bootstrap coincidiu com `1 - exp(-1)` dentro da tolerância.
- Bagging manual agregou probabilidades de árvores treinadas em réplicas distintas.
- Todos os modelos foram comparados nos mesmos folds.
- `max_features` e `min_samples_leaf` foram escolhidos sem consultar o teste.
- Correlação, OOB, saturação e estabilidade foram investigados apenas no desenvolvimento.
- O teste foi aberto uma única vez para comparadores congelados.

**O que não se pode concluir:** que Random Forest vencerá em todo dataset; que OOB é válido para tempo ou grupos; que MDI mede causalidade; ou que desacordo entre árvores captura todo tipo de incerteza. Na próxima aula, o ensemble deixará de ser independente e passará a corrigir erros sequencialmente.